In [52]:
import xml.etree.ElementTree as ET
import pandas as pd
xml_file = "cwec_v4.16.xml"
namespace = {'cwe': "http://cwe.mitre.org/cwe-7"}
root = ET.parse(xml_file).getroot()
root.attrib

{'Name': 'CWE',
 'Version': '4.16',
 'Date': '2024-11-19',
 '{http://www.w3.org/2001/XMLSchema-instance}schemaLocation': 'http://cwe.mitre.org/cwe-7 http://cwe.mitre.org/data/xsd/cwe_schema_v7.2.xsd'}

In [53]:
weaknesses = root.findall(".//cwe:Weaknesses/cwe:Weakness", namespace)
print(f"Found {len(weaknesses)} CWE entries.")


Found 965 CWE entries.


In [54]:
for weakness in weaknesses[:5]:  # Print first 5 for inspection
    cwe_id = weakness.get("ID", "N/A")
    name = weakness.get("Name", "N/A")
    abstraction = weakness.get("Abstraction", "N/A")
    structure = weakness.get("Structure", "N/A")
    status = weakness.get("Status", "N/A")

    print(f"CWE-{cwe_id}: {name}, Abstraction={abstraction}, Structure={structure}, Status={status}")


CWE-1004: Sensitive Cookie Without 'HttpOnly' Flag, Abstraction=Variant, Structure=Simple, Status=Incomplete
CWE-1007: Insufficient Visual Distinction of Homoglyphs Presented to User, Abstraction=Base, Structure=Simple, Status=Incomplete
CWE-102: Struts: Duplicate Validation Forms, Abstraction=Variant, Structure=Simple, Status=Incomplete
CWE-1021: Improper Restriction of Rendered UI Layers or Frames, Abstraction=Base, Structure=Simple, Status=Incomplete
CWE-1022: Use of Web Link to Untrusted Target with window.opener Access, Abstraction=Variant, Structure=Simple, Status=Incomplete


In [55]:
for weakness in weaknesses[:5]:  # Inspect first 5
    description = weakness.find("cwe:Description", namespace)
    extended_description = weakness.find("cwe:Extended_Description", namespace)

    desc_text = description.text.strip() if description is not None and description.text else "N/A"
    ext_desc_text = extended_description.text.strip() if extended_description is not None and extended_description.text else "N/A"

    print(f"CWE-{weakness.get('ID')}: {desc_text[:100]}...")  # Print first 100 characters


CWE-1004: The product uses a cookie to store sensitive information, but the cookie is not marked with the Http...
CWE-1007: The product displays information or identifiers to a user, but the display mechanism does not make i...
CWE-102: The product uses multiple validation forms with the same name, which might cause the Struts Validato...
CWE-1021: The web application does not restrict or incorrectly restricts frame objects or UI layers that belon...
CWE-1022: The web application produces links to untrusted external sites outside of its sphere of control, but...


In [56]:
for weakness in weaknesses[:5]:
    # Find Extended_Description
    extended_description_element = weakness.find(".//cwe:Extended_Description", namespace)

    if extended_description_element is not None:
        # Extract all child elements (handles possible inline XHTML tags)
        extended_desc = [paragraph.text.strip() for paragraph in extended_description_element if paragraph.text]
        
        # Join paragraphs into one string
        extended_description_text = " ".join(extended_desc) if extended_desc else "N/A"
    else:
        extended_description_text = "N/A"

    print(f"CWE-{weakness.get('ID')}: Extended Description -> {extended_description_text}")


CWE-1004: Extended Description -> N/A
CWE-1007: Extended Description -> Some glyphs, pictures, or icons can be semantically distinct to a program, while appearing very similar or identical to a human user. These are referred to as homoglyphs. For example, the lowercase "l" (ell) and uppercase "I" (eye) have different character codes, but these characters can be displayed in exactly the same way to a user, depending on the font. This can also occur between different character sets. For example, the Latin capital letter "A" and the Greek capital letter "Α" (Alpha) are treated as distinct by programs, but may be displayed in exactly the same way to a user. Accent marks may also cause letters to appear very similar, such as the Latin capital letter grave mark "À" and its equivalent "Á" with the acute accent. Adversaries can exploit this visual similarity for attacks such as phishing, e.g. by providing a link to an attacker-controlled hostname that looks like a hostname that the victim trus

In [71]:


# Extract attributes for the first 5 weaknesses
for weakness in weaknesses[:5]:
    cwe_id = weakness.get("ID", "N/A")  # CWE ID of the weakness
    
    # DEBUG: Print Weakness ID
    print(f"\nProcessing CWE-{cwe_id}...")

    # Look for the <Related_Weaknesses> section
    related_weaknesses_section = weakness.find("cwe:Related_Weaknesses", namespace)

    related_weaknesses = []
    
    if related_weaknesses_section is not None:
        for rw in related_weaknesses_section.findall("cwe:Related_Weakness", namespace):
            nature = rw.get("Nature", "N/A")
            related_cwe_id = rw.get("CWE_ID", "N/A")
            view_id = rw.get("View_ID", "N/A")
            ordinal = rw.get("Ordinal", "N/A")

            # DEBUG: Print each related weakness found
            print(f"  Found Related Weakness -> CWE-{related_cwe_id}, Nature: {nature}, View ID: {view_id}, Ordinal: {ordinal}")

            related_weaknesses.append(f"CWE-{related_cwe_id} ({nature})")

    related_text = ", ".join(related_weaknesses) if related_weaknesses else "N/A"
    
    print(f"CWE-{cwe_id}: Related Weaknesses -> {related_text}")



Processing CWE-1004...
  Found Related Weakness -> CWE-732, Nature: ChildOf, View ID: 1000, Ordinal: Primary
CWE-1004: Related Weaknesses -> CWE-732 (ChildOf)

Processing CWE-1007...
  Found Related Weakness -> CWE-451, Nature: ChildOf, View ID: 1000, Ordinal: Primary
CWE-1007: Related Weaknesses -> CWE-451 (ChildOf)

Processing CWE-102...
  Found Related Weakness -> CWE-694, Nature: ChildOf, View ID: 1000, Ordinal: Primary
  Found Related Weakness -> CWE-1173, Nature: ChildOf, View ID: 1000, Ordinal: N/A
  Found Related Weakness -> CWE-20, Nature: ChildOf, View ID: 700, Ordinal: Primary
CWE-102: Related Weaknesses -> CWE-694 (ChildOf), CWE-1173 (ChildOf), CWE-20 (ChildOf)

Processing CWE-1021...
  Found Related Weakness -> CWE-441, Nature: ChildOf, View ID: 1000, Ordinal: Primary
  Found Related Weakness -> CWE-610, Nature: ChildOf, View ID: 1003, Ordinal: Primary
  Found Related Weakness -> CWE-451, Nature: ChildOf, View ID: 1000, Ordinal: N/A
CWE-1021: Related Weaknesses -> CWE-441

In [58]:
for weakness in weaknesses[:5]:
    # Extract Weakness Ordinality
    ordinalities = [
        ordinality.text.strip() for ordinality in weakness.findall(".//cwe:Weakness_Ordinalities/cwe:Weakness_Ordinality/cwe:Ordinality", namespace)
        if ordinality is not None and ordinality.text
    ]

    # Join multiple ordinalities if they exist
    ordinalities_text = ", ".join(ordinalities) if ordinalities else "N/A"

    print(f"CWE-{weakness.get('ID')}: Weakness Ordinality -> {ordinalities_text}")


CWE-1004: Weakness Ordinality -> N/A
CWE-1007: Weakness Ordinality -> Resultant
CWE-102: Weakness Ordinality -> Primary
CWE-1021: Weakness Ordinality -> N/A
CWE-1022: Weakness Ordinality -> N/A


In [59]:
for weakness in weaknesses[:5]:
    # Extract platforms safely
    platforms = [
        p.get("Class", "N/A") for p in weakness.findall(".//cwe:Applicable_Platforms/cwe:Language", namespace) if p.get("Class")
    ] + [
        p.get("Class", "N/A") for p in weakness.findall(".//cwe:Applicable_Platforms/cwe:Technology", namespace) if p.get("Class")
    ] + [
        p.get("Name", "N/A") for p in weakness.findall(".//cwe:Applicable_Platforms/cwe:Language", namespace) if p.get("Name")
    ]

    # Remove duplicates and join values
    platforms_text = ", ".join(set(platforms)) if platforms else "N/A"

    print(f"CWE-{weakness.get('ID')}: Platforms -> {platforms_text}")


CWE-1004: Platforms -> Not Language-Specific, Web Based
CWE-1007: Platforms -> Not Language-Specific, Web Based
CWE-102: Platforms -> Java
CWE-1021: Platforms -> Web Based
CWE-1022: Platforms -> JavaScript, Web Based


In [74]:
for weakness in weaknesses[:5]:
    # Extract Alternate Terms
    alternate_terms = []
    for term in weakness.findall(".//cwe:Alternate_Terms/cwe:Alternate_Term/cwe:Term", namespace):
        if term is not None and term.text:
            alternate_terms.append(term.text.strip())
    
    # Extract Term Descriptions (if available)
    term_descriptions = []
    for alt_term in weakness.findall(".//cwe:Alternate_Terms/cwe:Alternate_Term", namespace):
        term_element = alt_term.find("cwe:Term", namespace)
        desc_element = alt_term.find("cwe:Description", namespace)
        
        if term_element is not None and term_element.text:
            term_text = term_element.text.strip()
            if desc_element is not None and desc_element.text:
                term_text += f" ({desc_element.text.strip()})"
            term_descriptions.append(term_text)
    
    # Join multiple terms with ", "
    alternate_terms_text = ", ".join(alternate_terms) if alternate_terms else "N/A"
    alternate_terms_with_desc_text = ", ".join(term_descriptions) if term_descriptions else "N/A"
    
    print(f"CWE-{weakness.get('ID')}: Alternate Terms -> {alternate_terms_text}")
    print(f"CWE-{weakness.get('ID')}: Alternate Terms with Descriptions -> {alternate_terms_with_desc_text}")

CWE-1004: Alternate Terms -> N/A
CWE-1004: Alternate Terms with Descriptions -> N/A
CWE-1007: Alternate Terms -> Homograph Attack
CWE-1007: Alternate Terms with Descriptions -> Homograph Attack ("Homograph" is often used as a synonym of "homoglyph" by researchers, but according to Wikipedia, a homograph is a word that has multiple, distinct meanings.)
CWE-102: Alternate Terms -> N/A
CWE-102: Alternate Terms with Descriptions -> N/A
CWE-1021: Alternate Terms -> Clickjacking, UI Redress Attack, Tapjacking
CWE-1021: Alternate Terms with Descriptions -> Clickjacking, UI Redress Attack, Tapjacking ("Tapjacking" is similar to clickjacking, except it is used for mobile applications in which the user "taps" the application instead of performing a mouse click.)
CWE-1022: Alternate Terms -> tabnabbing
CWE-1022: Alternate Terms with Descriptions -> tabnabbing


In [61]:
for weakness in weaknesses[:5]:
    # Extract Background Details
    background_details = [
        detail.text.strip() for detail in weakness.findall(".//cwe:Background_Details/cwe:Background_Detail", namespace)
        if detail is not None and detail.text
    ]

    # Join multiple background details if they exist
    background_details_text = " ".join(background_details) if background_details else "N/A"

    print(f"CWE-{weakness.get('ID')}: Background Details -> {background_details_text}")


CWE-1004: Background Details -> An HTTP cookie is a small piece of data attributed to a specific website and stored on the user's computer by the user's web browser. This data can be leveraged for a variety of purposes including saving information entered into form fields, recording user activity, and for authentication purposes. Cookies used to save or record information generated by the user are accessed and modified by script code embedded in a web page. While cookies used for authentication are created by the website's server and sent to the user to be attached to future requests. These authentication cookies are often not meant to be accessed by the web page sent to the user, and are instead just supposed to be attached to future requests to verify authentication details.
CWE-1007: Background Details -> N/A
CWE-102: Background Details -> N/A
CWE-1021: Background Details -> N/A
CWE-1022: Background Details -> N/A


In [62]:
for weakness in weaknesses[:5]:
    # Extract Modes of Introduction (Phase + Note)
    modes_of_intro = []
    for intro in weakness.findall(".//cwe:Modes_Of_Introduction/cwe:Introduction", namespace):
        phase = intro.find("cwe:Phase", namespace)
        note = intro.find("cwe:Note", namespace)

        phase_text = phase.text.strip() if phase is not None and phase.text else "N/A"
        note_text = note.text.strip() if note is not None and note.text else "N/A"

        modes_of_intro.append(f"{phase_text}: {note_text}")

    # Join multiple introductions with " | "
    modes_of_intro_text = " | ".join(modes_of_intro) if modes_of_intro else "N/A"

    print(f"CWE-{weakness.get('ID')}: Modes of Introduction -> {modes_of_intro_text}")


CWE-1004: Modes of Introduction -> Implementation: N/A
CWE-1007: Modes of Introduction -> Architecture and Design: This weakness may occur when characters from various character sets are allowed to be interchanged within a URL, username, email address, etc. without any notification to the user or underlying system being used. | Implementation: N/A
CWE-102: Modes of Introduction -> Implementation: N/A
CWE-1021: Modes of Introduction -> Implementation: N/A
CWE-1022: Modes of Introduction -> Architecture and Design: This weakness is introduced during the design of an application when the architect does not specify that a linked external document should not be able to alter the location of the calling page. | Implementation: This weakness is introduced during the coding of an application when the developer does not include the noopener and/or noreferrer value for the rel attribute.


In [63]:
for weakness in weaknesses[:5]:
    # Extract Likelihood of Exploit
    likelihood_element = weakness.find("cwe:Likelihood_Of_Exploit", namespace)
    likelihood_text = likelihood_element.text.strip() if likelihood_element is not None and likelihood_element.text else "N/A"

    print(f"CWE-{weakness.get('ID')}: Likelihood of Exploit -> {likelihood_text}")


CWE-1004: Likelihood of Exploit -> Medium
CWE-1007: Likelihood of Exploit -> Medium
CWE-102: Likelihood of Exploit -> N/A
CWE-1021: Likelihood of Exploit -> N/A
CWE-1022: Likelihood of Exploit -> Medium


In [64]:
for weakness in weaknesses[:5]:
    # Extract Common Consequences
    consequences_list = []
    for consequence in weakness.findall(".//cwe:Common_Consequences/cwe:Consequence", namespace):
        scopes = [scope.text.strip() for scope in consequence.findall("cwe:Scope", namespace) if scope.text]
        impacts = [impact.text.strip() for impact in consequence.findall("cwe:Impact", namespace) if impact.text]
        note = consequence.find("cwe:Note", namespace)

        scope_text = ", ".join(scopes) if scopes else "N/A"
        impact_text = ", ".join(impacts) if impacts else "N/A"
        note_text = note.text.strip() if note is not None and note.text else "N/A"

        consequences_list.append(f"Scope: {scope_text} | Impact: {impact_text} | Note: {note_text}")

    # Join multiple consequences with " || "
    consequences_text = " || ".join(consequences_list) if consequences_list else "N/A"

    print(f"CWE-{weakness.get('ID')}: Common Consequences -> {consequences_text}")


CWE-1004: Common Consequences -> Scope: Confidentiality | Impact: Read Application Data | Note: If the HttpOnly flag is not set, then sensitive information stored in the cookie may be exposed to unintended parties. || Scope: Integrity | Impact: Gain Privileges or Assume Identity | Note: If the cookie in question is an authentication cookie, then not setting the HttpOnly flag may allow an adversary to steal authentication data (e.g., a session ID) and assume the identity of the user.
CWE-1007: Common Consequences -> Scope: Integrity, Confidentiality | Impact: Other | Note: An attacker may ultimately redirect a user to a malicious website, by deceiving the user into believing the URL they are accessing is a trusted domain. However, the attack can also be used to forge log entries by using homoglyphs in usernames. Homoglyph manipulations are often the first step towards executing advanced attacks such as stealing a user's credentials, Cross-Site Scripting (XSS), or log forgery. If an atta

In [65]:
for weakness in weaknesses[:5]:
    # Extract Detection Methods
    detection_methods = []
    for method in weakness.findall(".//cwe:Detection_Methods/cwe:Detection_Method", namespace):
        method_name = method.find("cwe:Method", namespace)
        description = method.find("cwe:Description", namespace)
        effectiveness = method.find("cwe:Effectiveness", namespace)

        method_text = method_name.text.strip() if method_name is not None and method_name.text else "N/A"
        description_text = description.text.strip() if description is not None and description.text else "N/A"
        effectiveness_text = effectiveness.text.strip() if effectiveness is not None and effectiveness.text else "N/A"

        detection_methods.append(f"Method: {method_text} | Effectiveness: {effectiveness_text} | Description: {description_text}")

    # Join multiple methods with " || "
    detection_methods_text = " || ".join(detection_methods) if detection_methods else "N/A"

    print(f"CWE-{weakness.get('ID')}: Detection Methods -> {detection_methods_text}")


CWE-1004: Detection Methods -> Method: Automated Static Analysis | Effectiveness: High | Description: Automated static analysis, commonly referred to as Static Application Security Testing (SAST), can find some instances of this weakness by analyzing source code (or binary/compiled code) without having to execute it. Typically, this is done by building a model of data flow and control flow, then searching for potentially-vulnerable patterns that connect "sources" (origins of input) with "sinks" (destinations where the data interacts with external components, a lower layer such as the OS, etc.)
CWE-1007: Detection Methods -> Method: Manual Dynamic Analysis | Effectiveness: Moderate | Description: If utilizing user accounts, attempt to submit a username that contains homoglyphs. Similarly, check to see if links containing homoglyphs can be sent via email, web browsers, or other mechanisms.
CWE-102: Detection Methods -> N/A
CWE-1021: Detection Methods -> Method: Automated Static Analysis 

In [66]:
for weakness in weaknesses[:5]:
    # Extract Potential Mitigations
    mitigations_list = []
    for mitigation in weakness.findall(".//cwe:Potential_Mitigations/cwe:Mitigation", namespace):
        phase = mitigation.find("cwe:Phase", namespace)
        description_element = mitigation.find("cwe:Description", namespace)

        phase_text = phase.text.strip() if phase is not None and phase.text else "N/A"

        # Extract all paragraph elements inside Description without assuming 'xhtml:p'
        description_texts = [desc.text.strip() for desc in description_element if desc is not None and desc.text]
        description_text = " ".join(description_texts) if description_texts else "N/A"

        mitigations_list.append(f"Phase: {phase_text} | Description: {description_text}")

    # Join multiple mitigations with " || "
    mitigations_text = " || ".join(mitigations_list) if mitigations_list else "N/A"

    print(f"CWE-{weakness.get('ID')}: Potential Mitigations -> {mitigations_text}")


CWE-1004: Potential Mitigations -> Phase: Implementation | Description: N/A
CWE-1007: Potential Mitigations -> Phase: Implementation | Description: Use a browser that displays Punycode for IDNs in the URL and status bars, or which color code various scripts in URLs. Due to the prominence of homoglyph attacks, several browsers now help safeguard against this attack via the use of Punycode. For example, Mozilla Firefox and Google Chrome will display IDNs as Punycode if top-level domains do not restrict which characters can be used in domain names or if labels mix scripts for different languages. || Phase: Implementation | Description: Use an email client that has strict filters and prevents messages that mix character sets to end up in a user's inbox. Certain email clients such as Google's GMail prevent the use of non-Latin characters in email addresses or in links contained within emails. This helps prevent homoglyph attacks by flagging these emails and redirecting them to a user's spam

In [67]:
for weakness in weaknesses[:5]:
    # Extract Demonstrative Examples
    examples_list = []
    
    for example in weakness.findall(".//cwe:Demonstrative_Examples/cwe:Demonstrative_Example", namespace):
        # Extract Intro_Text
        intro_text = example.find("cwe:Intro_Text", namespace)
        intro_text_str = intro_text.text.strip() if intro_text is not None and intro_text.text else "N/A"

        # Extract Body_Text (multiple possible)
        body_texts = [bt.text.strip() for bt in example.findall("cwe:Body_Text", namespace) if bt.text]
        body_text_str = " ".join(body_texts) if body_texts else "N/A"

        # Extract Example_Code from all child elements
        example_code_element = example.find("cwe:Example_Code", namespace)
        example_code_str = " ".join(
            [elem.text.strip() for elem in example_code_element if elem.text]
        ) if example_code_element is not None else "N/A"

        # Store extracted data as a formatted string
        examples_list.append(f"Intro: {intro_text_str} | Code: {example_code_str} | Body: {body_text_str}")

    # Join multiple Demonstrative Examples with " || " to keep them separate in output
    examples_text = " || ".join(examples_list) if examples_list else "N/A"

    print(f"CWE-{weakness.get('ID')}: Demonstrative Examples -> {examples_text}")


CWE-1004: Demonstrative Examples -> Intro: In this example, a cookie is used to store a session ID for a client's interaction with a website. The intention is that the cookie will be sent to the website with each request made by the client. | Code: String sessionID = generateSessionId(); | Body: The snippet of code below establishes a new cookie to hold the sessionID. The HttpOnly flag is not set for the cookie. An attacker who can perform XSS could insert malicious script such as: When the client loads and executes this script, it makes a request to the attacker-controlled web site. The attacker can then log the request and steal the cookie. To mitigate the risk, use the setHttpOnly(true) method.
CWE-1007: Demonstrative Examples -> Intro: The following looks like a simple, trusted URL that a user may frequently access. | Code: http://www.еxаmрlе.соm | Body: However, the URL above is comprised of Cyrillic characters that look identical to the expected ASCII characters. This results in 

In [68]:
for weakness in weaknesses[:5]:
    # Extract Observed Examples
    observed_list = []
    
    for observed in weakness.findall(".//cwe:Observed_Examples/cwe:Observed_Example", namespace):
        reference = observed.find("cwe:Reference", namespace)
        description = observed.find("cwe:Description", namespace)
        link = observed.find("cwe:Link", namespace)

        reference_text = reference.text.strip() if reference is not None and reference.text else "N/A"
        description_text = description.text.strip() if description is not None and description.text else "N/A"
        link_text = link.text.strip() if link is not None and link.text else "N/A"

        observed_list.append(f"Reference: {reference_text} | Description: {description_text} | Link: {link_text}")

    # Join multiple observed examples with " || "
    observed_text = " || ".join(observed_list) if observed_list else "N/A"

    print(f"CWE-{weakness.get('ID')}: Observed Examples -> {observed_text}")


CWE-1004: Observed Examples -> Reference: CVE-2022-24045 | Description: Web application for a room automation system has client-side Javascript that sets a sensitive cookie without the HTTPOnly security attribute, allowing the cookie to be accessed. | Link: https://www.cve.org/CVERecord?id=CVE-2022-24045 || Reference: CVE-2014-3852 | Description: CMS written in Python does not include the HTTPOnly flag in a Set-Cookie header, allowing remote attackers to obtain potentially sensitive information via script access to this cookie. | Link: https://www.cve.org/CVERecord?id=CVE-2014-3852 || Reference: CVE-2015-4138 | Description: Appliance for managing encrypted communications does not use HttpOnly flag. | Link: https://www.cve.org/CVERecord?id=CVE-2015-4138
CWE-1007: Observed Examples -> Reference: CVE-2013-7236 | Description: web forum allows impersonation of users with homoglyphs in account names | Link: https://www.cve.org/CVERecord?id=CVE-2013-7236 || Reference: CVE-2012-0584 | Descript

In [69]:
for weakness in weaknesses[:5]:
    # Extract Related Attack Patterns (CAPEC IDs)
    attack_patterns = [
        rap.get("CAPEC_ID", "N/A")
        for rap in weakness.findall(".//cwe:Related_Attack_Patterns/cwe:Related_Attack_Pattern", namespace)
    ]

    # Join multiple CAPEC IDs with ", "
    attack_patterns_text = ", ".join(attack_patterns) if attack_patterns else "N/A"

    print(f"CWE-{weakness.get('ID')}: Related Attack Patterns -> {attack_patterns_text}")


CWE-1004: Related Attack Patterns -> N/A
CWE-1007: Related Attack Patterns -> 632
CWE-102: Related Attack Patterns -> N/A
CWE-1021: Related Attack Patterns -> 103, 181, 222, 504, 506, 587, 654
CWE-1022: Related Attack Patterns -> N/A


In [70]:
for weakness in weaknesses[:5]:
    # Extract Mapping Notes
    mapping_notes = weakness.find(".//cwe:Mapping_Notes", namespace)
    
    if mapping_notes is not None:
        usage = mapping_notes.find("cwe:Usage", namespace)
        rationale = mapping_notes.find("cwe:Rationale", namespace)
        comments = mapping_notes.find("cwe:Comments", namespace)
        
        # Extract multiple reasons
        reasons = [
            reason.get("Type", "N/A")
            for reason in mapping_notes.findall(".//cwe:Reasons/cwe:Reason", namespace)
        ]
        reasons_text = ", ".join(reasons) if reasons else "N/A"

        usage_text = usage.text.strip() if usage is not None and usage.text else "N/A"
        rationale_text = rationale.text.strip() if rationale is not None and rationale.text else "N/A"
        comments_text = comments.text.strip() if comments is not None and comments.text else "N/A"

        mapping_notes_text = f"Usage: {usage_text} | Rationale: {rationale_text} | Comments: {comments_text} | Reasons: {reasons_text}"
    else:
        mapping_notes_text = "N/A"

    print(f"CWE-{weakness.get('ID')}: Mapping Notes -> {mapping_notes_text}")


CWE-1004: Mapping Notes -> Usage: Allowed | Rationale: This CWE entry is at the Variant level of abstraction, which is a preferred level of abstraction for mapping to the root causes of vulnerabilities. | Comments: Carefully read both the name and description to ensure that this mapping is an appropriate fit. Do not try to 'force' a mapping to a lower-level Base/Variant simply to comply with this preferred level of abstraction. | Reasons: Acceptable-Use
CWE-1007: Mapping Notes -> Usage: Allowed | Rationale: This CWE entry is at the Base level of abstraction, which is a preferred level of abstraction for mapping to the root causes of vulnerabilities. | Comments: Carefully read both the name and description to ensure that this mapping is an appropriate fit. Do not try to 'force' a mapping to a lower-level Base/Variant simply to comply with this preferred level of abstraction. | Reasons: Acceptable-Use
CWE-102: Mapping Notes -> Usage: Allowed | Rationale: This CWE entry is at the Variant 